# نتایج فاز ۲: ارزش ویژگی‌های رادیومیکس

این notebook را **بعد از اجرای فاز ۲** باز کنید (با دادهٔ مصنوعی: `python tools/run_demo.py --phase2` یا دوبار کلیک روی `setup_windows.bat`).
هر cell را با `Shift+Enter` اجرا کنید. هر بخش به یکی از اهداف یا فرضیه‌های پروپوزال مربوط است.

> اگر در بالای VS Code از شما kernel خواست، **`.venv`** را انتخاب کنید، نه Python اصلی ویندوز.

> توضیح کامل هر قدم و معنی هر عدد: `docs/PROJECT_GUIDE_FA.md`

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = ROOT / "results"
if not (RESULTS / "analysis_table.csv").exists():
    raise SystemExit("نتایج فاز ۲ پیدا نشد. اول این را اجرا کنید:  python tools/run_demo.py --phase2")
pd.set_option("display.width", 160)
print((RESULTS / "analysis_table_report.txt").read_text())

## ۱. H1 (هدف اختصاصی ۱): کدام ویژگی‌ها با پیامدها ارتباط دارند؟

هر سطر یک ویژگی رادیومیکس است. `estimate` نسبت خطر (HR) یا نسبت شانس (OR) به‌ازای یک انحراف معیار است و `q_fdr` مقدار p اصلاح‌شده برای آزمون‌های متعدد. **H1 پذیرفته می‌شود اگر دست‌کم یک ویژگی q < 0.05 داشته باشد.**

In [ ]:
h1 = pd.read_csv(RESULTS / "h1_univariable.csv")
print(h1.groupby("outcome")["q_fdr"].apply(lambda q: (q < 0.05).sum()).rename("features with FDR < 0.05"))
h1[h1.q_fdr < 0.05].groupby("outcome").head(3).round(3)

## ۲. هدف اختصاصی ۲: رادیومیکس به‌تنهایی و همراه متغیرهای مرجع

سه گروه پیش‌بین مقایسه می‌شوند: **reference** (فقط SUVmean، PSMA-TV، LDH، Hb)، **radiomics** (فقط ویژگی‌های رادیومیکس) و **combined**. اگر میله‌های radiomics و combined بلندتر از reference باشند، رادیومیکس ارزش پیش‌بینی دارد.

In [ ]:
cv = pd.read_csv(RESULTS / "models_cv.csv")
main = cv[cv.metric.isin(["uno_c", "auc"])]
fig, axes = plt.subplots(1, main.endpoint.nunique(), figsize=(14, 4), sharey=True)
for ax, (ep, g) in zip(axes, main.groupby("endpoint")):
    p = g.pivot(index="model", columns="predictor_set", values="mean")[["reference", "radiomics", "combined"]]
    p.plot.bar(ax=ax, rot=0, color=["#999999", "#7F007F", "#00873E"])
    ax.set_title(f"{ep} ({g.metric.iloc[0]})"); ax.axhline(0.5, ls=":", c="k"); ax.set_ylim(0.4, 0.9)
plt.tight_layout(); plt.show()
main.pivot_table(index=["endpoint", "model"], columns="predictor_set", values="mean").round(3)

## ۳. اعتبارسنجی: bootstrap، temporal و leave-one-center-out

`*_corrected` عملکرد اصلاح‌شده برای optimism است. ردیف `temporal` مدل را روی ۳۰٪ بیماران جدیدتر می‌سنجد.

In [ ]:
f = sorted(RESULTS.glob("validation_*.csv"))[0]
val = pd.read_csv(f)
cols = [c for c in ["predictor_set", "analysis", "n", "n_val", "uno_c_corrected", "uno_c", "auc_corrected", "auc"] if c in val]
print(f.name); val[cols].round(3)

## ۴. H2 (هدف اختصاصی ۳): رادیومیکس چقدر به مدل مرجع اضافه می‌کند؟

`delta_c_corrected` افزایش C-index مدل combined نسبت به مدل مرجع است. **H2 برای OS پذیرفته می‌شود اگر این مقدار ≥ 0.03 باشد و CI از صفر بالاتر باشد.** نمودار decision curve نشان می‌دهد آیا مدل ترکیبی در تصمیم‌گیری بالینی سود بیشتری دارد.

In [ ]:
h2 = pd.read_csv(RESULTS / "h2_incremental_value.csv")
display(h2.drop(columns=[c for c in h2 if c == "note"]).round(4))
dca = pd.read_csv(RESULTS / "h2_decision_curve.csv")
fig, axes = plt.subplots(1, dca.endpoint.nunique(), figsize=(12, 4))
for ax, (ep, g) in zip(axes, dca.groupby("endpoint")):
    for m, gg in g.groupby("model"):
        ax.plot(gg.threshold, gg.net_benefit_model, label=m)
    ax.plot(gg.threshold, gg.net_benefit_treat_all, "k:", label="treat all"); ax.axhline(0, c="k", lw=0.8, label="treat none")
    ax.set_ylim(-0.05, max(0.05, g.net_benefit_model.max() * 1.1)); ax.set_title(f"{ep}: decision curve at 12 months")
    ax.set_xlabel("threshold probability"); ax.set_ylabel("net benefit"); ax.legend()
plt.tight_layout(); plt.show()

## ۵. H3: مهم‌ترین ویژگی‌های رادیومیکس و پایداری آن‌ها

میله‌ها اهمیت SHAP هستند (بنفش = رادیومیکس، خاکستری = متغیر مرجع). عدد کنار هر ویژگی رادیومیکس نشان می‌دهد در چند درصد نمونه‌های bootstrap انتخاب شده؛ **H3 به حداقل ۷۰٪ نیاز دارد**، به‌علاوهٔ تأیید دو پزشک طب هسته‌ای.

In [ ]:
imp = pd.read_csv(sorted(RESULTS.glob("h3_importance_*.csv"))[0], index_col=0).head(12)[::-1]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(imp.index, imp.shap_mean_abs, color=["#7F007F" if r else "#999999" for r in imp.is_radiomic])
for i, (v, fr) in enumerate(zip(imp.shap_mean_abs, imp.selection_frequency)):
    if fr == fr:
        ax.text(v, i, f"  {fr:.0%}", va="center")
ax.set_xlabel("mean |SHAP|"); ax.set_title("Most important features (combined model)")
plt.tight_layout(); plt.show()
pd.read_csv(sorted(RESULTS.glob("h3_direction_*.csv"))[0])